### Faster Whisper의 레이턴시를 Sample Rate별로 측정함

In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/modules/python-utils",
    "/workspaces/dev/modules/ai-utils",
    "/workspaces/dev/test/performance_test/esic",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path

In [ ]:
from sj_ai_utils.datasets.esic_v1 import search_all_data, search_file_from_dir
from sj_utils.audio import segment_audio
from sj_utils.evaluator import TimeChecker
from sj_utils.file.json import JsonSaver

In [ ]:
from util import load_mp4

In [ ]:
DESCRIPTION = """
ESIC dev 셋 기준 각 샘플레이트별 latency 측정
RTX4070 기준
"""

In [ ]:
SOURCE = "/workspaces/dev/datasets/ESIC-v1.1/v1.1/dev/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
OUTPUT = "/workspaces/dev/test/performance_test/esic/output/latency_test/20250722"

In [ ]:
MIN_SEGMENT = 16000
MAX_SEGMENT = 30 * 16000
STEP_SEGMENT = 16000

In [ ]:
yaml_saver = JsonSaver(DESCRIPTION)

In [ ]:
src = Path(SOURCE)
output = Path(OUTPUT)

In [ ]:
data_paths = search_all_data(src)[:2]
len(data_paths)

In [ ]:
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

In [ ]:
for segment_length in range(MIN_SEGMENT, MAX_SEGMENT + 1, STEP_SEGMENT):
    save_path = output / f"{segment_length}.json"
    transcribe_time = TimeChecker()
    for data_path in data_paths:
        audio_path = search_file_from_dir(data_path, "mp4")
        if audio_path is None:
            continue
        audio, _ = load_mp4(audio_path, sr=SAMPLE_RATE)

        for segment in segment_audio(audio, mean=segment_length, std=0, ratio=0):
            if len(segment) != segment_length:
                continue
            transcribe_time.start()
            segments, _ = model.transcribe(
                segment,
                beam_size=5,
                temperature=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
                language="en",
                word_timestamps=True,
            )
            for segment in segments:
                pass
            transcribe_time.check()
    result = {
        "segment_length": segment_length,
        "latency": transcribe_time.metric()
    }
    yaml_saver.save(result, save_path)